<a href="https://colab.research.google.com/github/AzaharaAED/Proyecto_ecosistema/blob/main/interfaz_nutrimind.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NutriMind - Interfaz provisional para el usuario
Este notebook genera y arranca la web en Google Colab.

In [ ]:
!pip -q install flask flask-cors

In [ ]:
!pip install -q "transformers==4.46.3" "accelerate==0.34.2" sentencepiece safetensors qwen-vl-utils huggingface_hub
!pip install -q "google-genai>=1.0.0"

In [ ]:
!jupyter nbconvert --to python "bascula.ipynb"
!jupyter nbconvert --to python "nevhor.ipynb"
!jupyter nbconvert --to python "comanda.ipynb"
!jupyter nbconvert --to python "cuestionario.ipynb"
!jupyter nbconvert --to python Sin_pip_Proyecto_modulo_general_gemini_personalizado.ipynb
!mv Sin_pip_Proyecto_modulo_general_gemini_personalizado.py modulo_general_gemini.py

[NbConvertApp] Converting notebook bascula.ipynb to python
[NbConvertApp] Writing 5385 bytes to bascula.py
[NbConvertApp] Converting notebook nevhor.ipynb to python
[NbConvertApp] Writing 17845 bytes to nevhor.py
[NbConvertApp] Converting notebook comanda.ipynb to python
[NbConvertApp] Writing 6191 bytes to comanda.py
[NbConvertApp] Converting notebook cuestionario.ipynb to python
[NbConvertApp] Writing 14350 bytes to cuestionario.py
[NbConvertApp] Converting notebook Sin_pip_Proyecto_modulo_general_gemini_personalizado.ipynb to python
[NbConvertApp] ERROR | Notebook JSON is invalid: Additional properties are not allowed ('errorDetails' was unexpected)

Failed validating 'additionalProperties' in error:

On instance['cells'][0]['outputs'][1]:
{'ename': 'ModuleNotFoundError',
 'errorDetails': {'actions': [{'action': 'open_url',
                               'actionText': 'Open Examples',
                               'url': '/notebooks/snippets/importing_libraries.ipynb'}]},
 'evalue'

In [ ]:
%%writefile fixed_app_colab.py
import os
import json
from flask import Flask, request, jsonify, send_from_directory
from flask_cors import CORS

import bascula
import cuestionario
import modulo_general_gemini as modulo_general

try:
    from google.colab import userdata
except Exception:
    userdata = None

try:
    import google.generativeai as legacy_genai
except Exception as e:
    raise RuntimeError(
        "No se pudo importar 'google.generativeai'. Ejecuta antes: !pip install -q google-generativeai"
    ) from e

modulo_general.genai = legacy_genai
modulo_general.json = json

app = Flask(__name__, static_folder=".", static_url_path="")
CORS(app, resources={r"/api/*": {"origins": "*"}})

TIPOS_PETICION = {
    "ver_progreso": "ver_progreso",
    "que_puedo_cocinar": "que_puedo_cocinar",
    "recomendar_receta": "recomendar_receta",
    "generar_dieta": "generar_dieta",
    "otra_consulta": "otra_consulta",
}

_estado_cuestionario = cuestionario.inicializar_modulo_cuestionario()
_usuarios_df = _estado_cuestionario.get("usuarios_df")
_perfiles_dict = _estado_cuestionario.get("perfiles_dict", {})

_estado_bascula = bascula.inicializar_modulo_bascula()
_bascula_df = _estado_bascula.get("bascula_df")

if _usuarios_df is None:
    raise RuntimeError("cuestionario.inicializar_modulo_cuestionario() no devolvió 'usuarios_df'.")

if _bascula_df is None:
    raise RuntimeError("bascula.inicializar_modulo_bascula() no devolvió 'bascula_df'.")


def _resolver_api_key() -> str:
    api_key = (
        os.environ.get("GEMINI_API_KEY", "").strip()
        or os.environ.get("GOOGLE_API_KEY", "").strip()
    )
    if api_key:
        return api_key

    if userdata is not None:
        for nombre in ("GEMINI_API_KEY", "GOOGLE_API_KEY"):
            try:
                valor = userdata.get(nombre)
                if valor and str(valor).strip():
                    valor = str(valor).strip()
                    os.environ[nombre] = valor
                    return valor
            except Exception:
                pass

    raise RuntimeError(
        "No se ha encontrado GEMINI_API_KEY ni GOOGLE_API_KEY ni en variables de entorno ni en los secretos de Colab."
    )


def _configurar_genai_legacy():
    api_key = _resolver_api_key()
    legacy_genai.configure(api_key=api_key)
    modulo_general.genai = legacy_genai
    modulo_general.json = json
    return api_key


def _normalizar_respuesta(respuesta):
    if respuesta is None:
        return ""
    if isinstance(respuesta, str):
        return respuesta.strip()
    texto = getattr(respuesta, "text", None)
    if isinstance(texto, str) and texto.strip():
        return texto.strip()
    return str(respuesta).strip()


@app.after_request
def add_headers(response):
    response.headers["Access-Control-Allow-Origin"] = "*"
    response.headers["Access-Control-Allow-Headers"] = (
        "Content-Type,Authorization,ngrok-skip-browser-warning,x-colab-notebook-cache-tag"
    )
    response.headers["Access-Control-Allow-Methods"] = "GET,POST,OPTIONS"
    return response


@app.route("/")
def index():
    return send_from_directory(".", "fixed_style_app_colab.html")


@app.route("/api/ping", methods=["GET"])
def ping():
    return jsonify({"status": "ok", "mode": "real"})


@app.route("/api/perfiles", methods=["GET"])
def perfiles():
    try:
        if hasattr(_usuarios_df, "columns") and "user_id" in _usuarios_df.columns:
            ids = [
                str(x).strip()
                for x in _usuarios_df["user_id"].dropna().astype(str).tolist()
                if str(x).strip()
            ]
            if not ids:
                raise RuntimeError("La columna 'user_id' existe pero no contiene perfiles válidos.")
            return jsonify({"perfiles": ids})

        if isinstance(_perfiles_dict, dict) and _perfiles_dict:
            ids = [str(k).strip() for k in _perfiles_dict.keys() if str(k).strip()]
            if not ids:
                raise RuntimeError("perfiles_dict existe pero no contiene claves válidas.")
            return jsonify({"perfiles": ids})

        raise RuntimeError("No se pudieron obtener perfiles.")
    except Exception as e:
        return jsonify({"error": f"Error al obtener perfiles: {e}"}), 500


@app.route("/api/consulta", methods=["POST", "OPTIONS"])
def consulta():
    if request.method == "OPTIONS":
        return jsonify({"ok": True})

    data = request.get_json(silent=True)
    if not data:
        return jsonify({"error": "El body de la petición está vacío o no es JSON válido."}), 400

    user_id = str(data.get("user_id", "")).strip()
    tipo_peticion = str(data.get("tipo_peticion", "otra_consulta")).strip()
    texto_usuario = str(data.get("texto_usuario", "")).strip()
    url_nevera = str(data.get("url_nevera", "")).strip()
    url_horno = str(data.get("url_horno", "")).strip()

    if not user_id:
        return jsonify({"error": "Falta el campo obligatorio: user_id"}), 400
    if not texto_usuario:
        return jsonify({"error": "Falta el campo obligatorio: texto_usuario"}), 400
    if tipo_peticion not in TIPOS_PETICION:
        return jsonify({"error": f"tipo_peticion no válido: {tipo_peticion}"}), 400

    try:
        api_key = _configurar_genai_legacy()

        if not hasattr(modulo_general, "ejecutar_asistente_personalizado_web"):
            raise RuntimeError(
                "Falta la función 'ejecutar_asistente_personalizado_web' en modulo_general_gemini."
            )

        resultado = modulo_general.ejecutar_asistente_personalizado_web(
            user_id=user_id,
            tipo_peticion=tipo_peticion,
            texto_usuario=texto_usuario,
            bascula_df=_bascula_df,
            api_key=api_key,
            model_name="gemini-2.5-flash",
            url_nevera=url_nevera or None,
            url_horno=url_horno or None,
        )

        if not isinstance(resultado, dict):
            raise RuntimeError("La función del asistente no devolvió un diccionario válido.")

        if resultado.get("estado") != "ok":
            raise RuntimeError(resultado.get("respuesta", "Error desconocido en el asistente."))

        respuesta_texto = _normalizar_respuesta(resultado.get("respuesta"))
        if not respuesta_texto:
            raise RuntimeError("Gemini devolvió una respuesta vacía.")

        return jsonify({
            "estado": "ok",
            "mode": "real",
            "user_id": resultado.get("user_id", user_id),
            "respuesta": respuesta_texto,
        })

    except Exception as e:
        return jsonify({"error": f"Error en /api/consulta: {e}"}), 500


if __name__ == "__main__":
    _configurar_genai_legacy()
    app.run(host="0.0.0.0", port=5000, debug=False, use_reloader=False)

Overwriting fixed_app_colab.py


In [ ]:
%%writefile fixed_style_app_colab.html
<!DOCTYPE html>
<html lang="es">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>NutriMind · Colab</title>
<link href="https://fonts.googleapis.com/css2?family=DM+Serif+Display:ital@0;1&family=DM+Sans:wght@300;400;500;700&display=swap" rel="stylesheet">
<style>
  :root {
    --cream:#F5F0E8; --warm-white:#FDFAF5; --charcoal:#1C1C1E; --mid:#6B6B6B;
    --light:#B0A898; --border:#E2DBD0; --green:#2D6A4F; --green-light:#E8F4EE;
    --green-mid:#52B788; --red:#C0392B; --red-light:#FDECEA;
    --card-shadow:0 1px 3px rgba(0,0,0,.06),0 4px 16px rgba(0,0,0,.04);
  }

  * { box-sizing:border-box; margin:0; padding:0; }

  body {
    font-family:'DM Sans',sans-serif;
    background:var(--cream);
    color:var(--charcoal);
    min-height:100vh;
    line-height:1.6;
  }

  .app {
    display:grid;
    grid-template-columns:300px 1fr;
    min-height:100vh;
  }

  .sidebar {
    background:var(--charcoal);
    color:#fff;
    display:flex;
    flex-direction:column;
    padding:2rem 1.5rem;
    position:sticky;
    top:0;
    height:100vh;
  }

  .sidebar-logo {
    display:flex;
    align-items:center;
    gap:12px;
    margin-bottom:2rem;
  }

  .logo-icon {
    width:46px;
    height:46px;
    object-fit:contain;
    border-radius:12px;
    display:block;
    background:#fff;
    padding:4px;
    box-shadow:0 2px 10px rgba(0,0,0,.18);
    flex-shrink:0;
  }

  .logo-text {
    font-family:'DM Serif Display',serif;
    font-size:20px;
    line-height:1.1;
  }

  .logo-sub {
    font-size:11px;
    color:var(--light);
    margin-top:2px;
  }

  .sidebar-section { margin-bottom:1.25rem; }

  .sidebar-label {
    font-size:10px;
    font-weight:700;
    letter-spacing:.1em;
    text-transform:uppercase;
    color:var(--light);
    margin-bottom:.6rem;
    display:block;
  }

  .hint {
    font-size:12px;
    color:rgba(255,255,255,.65);
    margin-top:8px;
  }

  .url-config {
    background:rgba(255,255,255,.05);
    border:1px solid rgba(255,255,255,.1);
    border-radius:10px;
    padding:10px 12px;
    margin-bottom:1.25rem;
  }

  .url-config input {
    width:100%;
    background:rgba(0,0,0,.25);
    border:1px solid rgba(255,255,255,.12);
    border-radius:6px;
    padding:9px;
    font-size:12px;
    color:#fff;
    outline:none;
  }

  .url-status {
    font-size:11px;
    margin-top:7px;
    display:flex;
    align-items:center;
    gap:5px;
  }

  .sdot {
    width:6px;
    height:6px;
    border-radius:50%;
    background:var(--light);
    flex-shrink:0;
  }

  .sdot.ok { background:var(--green-mid); }
  .sdot.err { background:#e55; }
  .sdot.chk { background:#F5A623; animation:blink .7s ease-in-out infinite; }

  @keyframes blink {
    0%,100% { opacity:1; }
    50% { opacity:.25; }
  }

  .profile-btn, .tipo-btn {
    width:100%;
    text-align:left;
    border-radius:10px;
    padding:10px 12px;
    color:#fff;
    font-size:14px;
    cursor:pointer;
  }

  .profile-btn {
    background:rgba(255,255,255,.06);
    border:1px solid rgba(255,255,255,.1);
  }

  .profile-btn.active {
    background:var(--green);
    border-color:var(--green-mid);
  }

  .tipo-btn {
    background:transparent;
    border:1px solid rgba(255,255,255,.1);
    color:rgba(255,255,255,.7);
    margin-bottom:4px;
  }

  .tipo-btn.selected {
    background:rgba(82,183,136,.2);
    border-color:var(--green-mid);
    color:var(--green-mid);
  }

  .sidebar-footer {
    margin-top:auto;
    font-size:11px;
    color:rgba(255,255,255,.25);
    line-height:1.5;
  }

  .main {
    display:flex;
    flex-direction:column;
    padding:2.5rem 2rem;
    max-width:840px;
  }

  .page-title {
    font-family:'DM Serif Display',serif;
    font-size:34px;
    line-height:1.2;
  }

  .page-title em {
    color:var(--green);
    font-style:normal;
  }

  .page-sub {
    font-size:14px;
    color:var(--mid);
    margin-top:4px;
    margin-bottom:1.5rem;
  }

  .card {
    background:var(--warm-white);
    border-radius:16px;
    border:1px solid var(--border);
    padding:1.5rem;
    margin-bottom:1.25rem;
    box-shadow:var(--card-shadow);
  }

  .card-title {
    font-size:11px;
    font-weight:700;
    letter-spacing:.08em;
    text-transform:uppercase;
    color:var(--light);
    margin-bottom:1rem;
  }

  .field {
    display:flex;
    flex-direction:column;
    gap:6px;
  }

  .url-row {
    display:grid;
    grid-template-columns:1fr 1fr;
    gap:12px;
  }

  input, textarea {
    background:var(--cream);
    border:1px solid var(--border);
    border-radius:12px;
    padding:12px;
    font-size:14px;
    outline:none;
    width:100%;
  }

  textarea {
    min-height:120px;
    resize:vertical;
  }

  .btn-submit {
    width:100%;
    background:var(--charcoal);
    color:#fff;
    border:none;
    border-radius:12px;
    padding:14px;
    font-size:15px;
    cursor:pointer;
    margin-top:.7rem;
  }

  .btn-submit:disabled {
    opacity:.65;
    cursor:not-allowed;
  }

  .badge {
    display:inline-flex;
    align-items:center;
    gap:6px;
    padding:6px 10px;
    border-radius:999px;
    font-size:12px;
    background:var(--green-light);
    color:var(--green);
    border:1px solid var(--green-mid);
    margin-bottom:1rem;
  }

  .response-card {
    background:var(--warm-white);
    border-radius:16px;
    border:1px solid var(--border);
    box-shadow:var(--card-shadow);
    overflow:hidden;
  }

  .response-header {
    display:flex;
    align-items:center;
    justify-content:space-between;
    padding:1rem 1.5rem;
    border-bottom:1px solid var(--border);
    background:var(--green-light);
    gap:12px;
  }

  .response-tag {
    font-size:11px;
    font-weight:700;
    letter-spacing:.08em;
    text-transform:uppercase;
    color:var(--green);
  }

  .response-meta {
    font-size:12px;
    color:var(--mid);
  }

  .response-body {
    padding:1.5rem;
    font-size:14px;
    line-height:1.8;
    white-space:pre-wrap;
    word-break:break-word;
  }

  .error-card {
    background:var(--red-light);
    border:1px solid #F5C6C2;
    border-radius:12px;
    padding:1rem 1.25rem;
    font-size:13px;
    color:var(--red);
  }

  @media (max-width: 760px) {
    .app { grid-template-columns:1fr; }
    .sidebar { position:static; height:auto; }
    .url-row { grid-template-columns:1fr; }
    .main { padding:1.5rem 1rem; }
  }
</style>
</head>
<body>
<div class="app">
  <aside class="sidebar">
    <div class="sidebar-logo">
      <img src="nutrimind.png" alt="Logo NutriMind" class="logo-icon">
      <div>
        <div class="logo-text">NutriMind</div>
        <div class="logo-sub">Versión Colab</div>
      </div>
    </div>

    <div class="url-config">
      <span class="sidebar-label">URL del servidor Colab</span>
      <input type="text" id="server-url-input" placeholder="https://....colab.dev">
      <div class="url-status">
        <span class="sdot" id="sdot"></span>
        <span id="stext" style="color:var(--light)">Sin configurar</span>
      </div>
      <div class="hint">Pega aquí la URL que imprime <code>google.colab.kernel.proxyPort(5000)</code></div>
    </div>

    <div class="sidebar-section">
      <span class="sidebar-label">Perfil activo</span>
      <div id="sidebar-profile"><button class="profile-btn" disabled>Primero conecta el servidor</button></div>
    </div>

    <div class="sidebar-section">
      <span class="sidebar-label">Tipo de consulta</span>
      <button class="tipo-btn selected" data-tipo="recomendar_receta">Recomendar receta</button>
      <button class="tipo-btn" data-tipo="que_puedo_cocinar">¿Qué puedo cocinar?</button>
      <button class="tipo-btn" data-tipo="generar_dieta">Generar dieta</button>
      <button class="tipo-btn" data-tipo="ver_progreso">Ver progreso</button>
      <button class="tipo-btn" data-tipo="otra_consulta">Otra consulta</button>
    </div>

    <div class="sidebar-footer">
      NutriMind · Flask · Colab proxy · Gemini<br>
      nevera · horno · báscula · cuestionario
    </div>
  </aside>

  <main class="main">
    <h1 class="page-title">Tu asistente de nutrición <em>NutriMind</em></h1>
    <p class="page-sub" id="page-sub">Conecta primero con la URL pública de Colab.</p>

    <div class="badge" id="mode-badge" style="display:none;">Servidor conectado</div>

    <div class="card">
      <div class="card-title">Imágenes opcionales</div>
      <div class="url-row">
        <div class="field">
          <label>URL de imagen de la nevera</label>
          <input type="url" id="url-nevera" placeholder="https://...">
        </div>
        <div class="field">
          <label>URL de imagen del horno</label>
          <input type="url" id="url-horno" placeholder="https://...">
        </div>
      </div>
    </div>

    <div class="card">
      <div class="card-title">Tu consulta</div>
      <textarea id="texto-usuario" placeholder="Ej. Quiero una cena ligera con lo que tengo en casa..."></textarea>
      <button class="btn-submit" id="btn-enviar">Enviar al asistente</button>
    </div>

    <div id="resultado"></div>
  </main>
</div>

<script>
const BASE_HEADERS = {
  "ngrok-skip-browser-warning": "true",
  "x-colab-notebook-cache-tag": "skip"
};

let SERVER_URL = "";
let selectedUserId = null;
let selectedTipo = "recomendar_receta";
let pingTimer = null;

function escHtml(s) {
  return String(s)
    .replace(/&/g,"&amp;")
    .replace(/</g,"&lt;")
    .replace(/>/g,"&gt;")
    .replace(/"/g,"&quot;");
}

function escAttr(s) {
  return String(s).replace(/'/g,"\\'");
}

function clean(url) {
  return url.trim().replace(/\/+$/, "");
}

function setStatus(cls, txt) {
  const dot = document.getElementById("sdot");
  const span = document.getElementById("stext");
  dot.className = "sdot" + (cls ? " " + cls : "");
  span.textContent = txt;
  span.style.color = cls === "ok" ? "var(--green-mid)" : cls === "err" ? "#e55" : "var(--light)";
}

document.getElementById("server-url-input").addEventListener("input", (e) => {
  clearTimeout(pingTimer);
  SERVER_URL = clean(e.target.value);

  if (!SERVER_URL) {
    setStatus("", "Sin configurar");
    return;
  }

  setStatus("chk", "Comprobando...");
  pingTimer = setTimeout(pingAndLoad, 500);
});

async function pingAndLoad() {
  try {
    const r = await fetch(`${SERVER_URL}/api/ping`, { headers: BASE_HEADERS });
    const d = await r.json();

    if (!r.ok) throw new Error(`HTTP ${r.status}`);

    setStatus("ok", `Servidor conectado ✓ (${d.mode || "ok"})`);
    document.getElementById("mode-badge").style.display = "inline-flex";
    document.getElementById("mode-badge").textContent = `Modo ${d.mode || "ok"}`;
    await cargarPerfiles();
  } catch (e) {
    setStatus("err", "No se puede conectar");
    document.getElementById("resultado").innerHTML = `<div class="error-card">No se pudo conectar con la URL del servidor. Revisa que Flask siga corriendo en Colab y que la URL sea la de <code>proxyPort(5000)</code>.</div>`;
  }
}

async function cargarPerfiles() {
  try {
    const r = await fetch(`${SERVER_URL}/api/perfiles`, { headers: BASE_HEADERS });
    const d = await r.json();
    const perfiles = d.perfiles || [];

    const html = perfiles.length
      ? perfiles.map(p => `<button class="profile-btn ${selectedUserId===p ? 'active' : ''}" onclick="selectProfile('${escAttr(p)}')">${escHtml(p)}</button>`).join("")
      : `<button class="profile-btn" disabled>No hay perfiles</button>`;

    document.getElementById("sidebar-profile").innerHTML = html;

    if (!selectedUserId && perfiles.length) {
      selectProfile(perfiles[0]);
    }
  } catch {
    document.getElementById("sidebar-profile").innerHTML = `<button class="profile-btn" disabled>Error cargando perfiles</button>`;
  }
}

function selectProfile(userId) {
  selectedUserId = userId;
  document.getElementById("page-sub").textContent = `Perfil activo: ${userId}`;

  document.querySelectorAll("#sidebar-profile .profile-btn").forEach(btn => {
    btn.classList.toggle("active", btn.textContent.trim() === userId);
  });
}

document.querySelectorAll(".tipo-btn").forEach(btn => {
  btn.addEventListener("click", () => {
    document.querySelectorAll(".tipo-btn").forEach(b => b.classList.remove("selected"));
    btn.classList.add("selected");
    selectedTipo = btn.dataset.tipo;
  });
});

document.getElementById("btn-enviar").addEventListener("click", enviarConsulta);

async function enviarConsulta() {
  if (!SERVER_URL) {
    alert("Introduce la URL del servidor de Colab.");
    return;
  }

  if (!selectedUserId) {
    alert("Selecciona un perfil.");
    return;
  }

  const texto = document.getElementById("texto-usuario").value.trim();

  if (!texto) {
    document.getElementById("texto-usuario").focus();
    return;
  }

  const btn = document.getElementById("btn-enviar");
  btn.disabled = true;
  btn.textContent = "Enviando...";

  const payload = {
    user_id: selectedUserId,
    tipo_peticion: selectedTipo,
    texto_usuario: texto,
    url_nevera: document.getElementById("url-nevera").value.trim(),
    url_horno: document.getElementById("url-horno").value.trim()
  };

  try {
    const r = await fetch(`${SERVER_URL}/api/consulta`, {
      method: "POST",
      headers: { ...BASE_HEADERS, "Content-Type": "application/json" },
      body: JSON.stringify(payload)
    });

    let d;
    try {
      d = await r.json();
    } catch {
      throw new Error("La respuesta no es JSON válido");
    }

    if (!r.ok || d.error) {
      throw new Error(d.error || `Error ${r.status}`);
    }

    const hora = new Date().toLocaleTimeString("es-ES", { hour:"2-digit", minute:"2-digit" });

    document.getElementById("resultado").innerHTML = `
      <div class="response-card">
        <div class="response-header">
          <span class="response-tag">${escHtml(selectedTipo)}</span>
          <span class="response-meta">${escHtml(selectedUserId)} · ${hora}</span>
        </div>
        <div class="response-body">${escHtml(d.respuesta || "")}</div>
      </div>`;
  } catch (e) {
    document.getElementById("resultado").innerHTML = `<div class="error-card">⚠ ${escHtml(e.message || "Error desconocido")}</div>`;
  } finally {
    btn.disabled = false;
    btn.textContent = "Enviar al asistente";
  }
}
</script>
</body>
</html>

Overwriting fixed_style_app_colab.html


In [ ]:
import threading, time
from google.colab.output import eval_js
from fixed_app_colab import app

def run():
    app.run(host="0.0.0.0", port=5000, debug=False, use_reloader=False)

threading.Thread(target=run, daemon=True).start()
time.sleep(2)

url = eval_js("google.colab.kernel.proxyPort(5000)")
print("URL pública del servidor:")
print(url)
print("\nAbre también esta URL en el navegador si quieres servir el HTML desde Flask:")
print(url)


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 16.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.4 MB/s eta 0:00:00
 * Serving Flask app 'fixed_app_colab'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


URL pública del servidor:
https://5000-m-s-kkb-usw1c2-2zgeqh9jpq1lv-c.us-west1-2.prod.colab.dev

Abre también esta URL en el navegador si quieres servir el HTML desde Flask:
https://5000-m-s-kkb-usw1c2-2zgeqh9jpq1lv-c.us-west1-2.prod.colab.dev
